In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# %pip install xgboost

In [0]:
%restart_python

In [0]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mlflow
import mlflow.sklearn

sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG
import src.functions as fn


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
gold_df_train = spark.read.table("workspace.fall_detection_project.gold_features_train").toPandas() # train_df

gold_df_test = spark.read.table("workspace.fall_detection_project.gold_features_test").toPandas()   # test_df

print(f"Train  : {gold_df_train.shape}")
print(f"Test   : {gold_df_test.shape}")


In [0]:
gold_df_train.columns

In [0]:
non_feature_cols = ["label_binary", "label_multiclass", "split", "signal_id"]
feature_names = [c for c in gold_df_train.columns if c not in non_feature_cols]

In [0]:
X_train = gold_df_train.drop(columns=['label_binary', 'label_multiclass', 'split', 'signal_id'])
y_train_bin = gold_df_train['label_binary']
y_train_mul = gold_df_train['label_multiclass']


X_test = gold_df_test.drop(columns=['label_binary', 'label_multiclass', 'split', 'signal_id'])
y_test_bin = gold_df_test['label_binary']
y_test_mul = gold_df_test['label_multiclass']

In [0]:
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Features: {len(feature_names)}")

In [0]:
mlflow.set_experiment(CFG["mlflow"]["experiment_name_binary"])

models = {
    "RandomForest"          : RandomForestClassifier(
        n_estimators=200, 
        max_depth=10, random_state=42, 
        class_weight="balanced"),
    
    "GradientBoosting"      : GradientBoostingClassifier(
        n_estimators=200, 
        learning_rate=0.1, max_depth=5, 
        random_state=42),
    
    "SVM"                   : SVC(
        kernel="rbf", 
        C=10, gamma="scale", 
        probability=True, class_weight="balanced"),
    
    "LogisticRegression"    : LogisticRegression(
        max_iter=1000, 
        class_weight="balanced", 
        random_state=42),
    
    "KNN"                   : KNeighborsClassifier(
        n_neighbors=5),
    
    "XGBoost"               : XGBClassifier(
        n_estimators=200, 
        learning_rate=0.1, 
        max_depth=6, 
        random_state=42, 
        eval_metric="logloss", 
        scale_pos_weight=4.48), # handles class imbalance — ratio of ADL/Fall
    
    "NeuralNetwork"         : MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        max_iter=500,
        random_state=42),
}

In [0]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

scaled_models = ["SVM", "LogisticRegression", "NeuralNetwork"]

le = LabelEncoder()
y_train_encoded_bin = le.fit_transform(y_train_bin)

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):

        X_tr = X_train_scaled if model_name in scaled_models else X_train

        # cross validation
        cv_scores = cross_val_score(model, X_tr, y_train_encoded_bin, cv=cv, scoring="f1")
        cv_mean   = cv_scores.mean()
        cv_std    = cv_scores.std()

        # log parameters
        mlflow.log_params(model.get_params())

        # train model
        model.fit(X_tr, y_train_encoded_bin)

        # predict on train
        y_pred_bin = model.predict(X_tr)
        train_f1   = f1_score(y_train_encoded_bin, y_pred_bin, pos_label=1)

        # log metrics
        mlflow.log_metric("cv_f1_mean", cv_mean)
        mlflow.log_metric("cv_f1_std",  cv_std)
        mlflow.log_metric("train_f1",   train_f1)

        # log model
        mlflow.sklearn.log_model(model, "model")

        print(f"{model_name} → CV F1: {cv_mean:.4f} ± {cv_std:.4f}  Train F1: {train_f1:.4f}")

In [0]:
y_train_encoded_mul = le.fit_transform(y_train_mul)

mlflow.set_experiment(CFG["mlflow"]["experiment_name_multiclass"])

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):

        X_tr = X_train_scaled if model_name in scaled_models else X_train

        # cross validation
        cv_scores_mul = cross_val_score(model, X_tr, y_train_encoded_mul, cv=cv, scoring="f1_macro")
        cv_mean_mul   = cv_scores_mul.mean()
        cv_std_mul    = cv_scores_mul.std()

        # log parameters
        mlflow.log_params(model.get_params())

        # train model
        model.fit(X_tr, y_train_encoded_mul)

        # predict on train
        y_pred_mul = model.predict(X_tr)
        train_f1_mul   = f1_score(y_train_encoded_mul, y_pred_mul, average="macro")

        # log metrics
        mlflow.log_metric("cv_f1_mean", cv_mean_mul)
        mlflow.log_metric("cv_f1_std",  cv_std_mul)
        mlflow.log_metric("train_f1",   train_f1_mul)

        # log model
        mlflow.sklearn.log_model(model, "model")

        print(f"{model_name} → CV F1: {cv_mean_mul:.4f} ± {cv_std_mul:.4f}  Train F1: {train_f1_mul:.4f}")